# RelaLeap Colab Phase 0 Comparison

This notebook is a thin Colab wrapper around the GitHub repo. The repo, configs, baseline, and run artifacts are the source of truth.

Before running, choose `Runtime -> Change runtime type -> GPU` if a GPU run is desired.

In [ ]:
!nvidia-smi || true
import torch, platform
print('platform:', platform.platform())
print('torch:', torch.__version__)
print('cuda_available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('cuda_device:', torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
repo = Path('/content/relaleap')
if repo.exists():
    %cd /content/relaleap
    !git pull --ff-only
else:
    %cd /content
    !git clone https://github.com/bgoertzel-sing/relaleap.git
    %cd /content/relaleap

In [ ]:
!python -m pip install -q -e '.[notebook]'
!python -m relaleap.experiments.compare --out results/comparisons/colab_phase0 --baseline-reference baselines/phase0_char_smoke_comparison.json

In [ ]:
import base64
import io
import json
import textwrap
import zipfile
from pathlib import Path
summary_path = Path('results/comparisons/colab_phase0/summary.json')
baseline_comparison_path = Path('results/comparisons/colab_phase0/baseline_comparison.json')
print(summary_path.read_text())
print(baseline_comparison_path.read_text())
summary = json.loads(summary_path.read_text())
baseline_comparison = json.loads(baseline_comparison_path.read_text())
assert summary['status'] == 'ok'
assert summary['verdict']['status'] == 'pass'
assert baseline_comparison['status'] == 'pass'
accepted = summary['verdict']['hep_alpha_acceptance']['accepted_alpha']
print('Accepted HEP alpha:', accepted)
artifact_root = Path('results/comparisons/colab_phase0')
bundle_buffer = io.BytesIO()
with zipfile.ZipFile(bundle_buffer, mode='w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(artifact_root.rglob('*')):
        if path.is_file():
            archive.write(path, path.as_posix())
bundle = base64.b64encode(bundle_buffer.getvalue()).decode('ascii')
print('RELALEAP_ARTIFACT_BUNDLE_ZIP_BASE64_BEGIN')
print('\n'.join(textwrap.wrap(bundle, width=120)))
print('RELALEAP_ARTIFACT_BUNDLE_ZIP_BASE64_END')
print('RelaLeap Colab Phase 0 comparison completed.')